# 3.07 Milvus Vector Store

**Milvus**는 대규모 프로덕션용 벡터 DB. 분산 모드(`milvus-distributed`)와 로컬 파일 모드(`milvus-lite`)를 제공한다. 노트북에서는 Milvus Lite로 외부 서비스 없이 실습한다.

## 학습 목표

- `pymilvus[milvus_lite]` 로 로컬 파일 DB 실습
- `Milvus` 클래스의 `connection_args={'uri': './db.sqlite'}` 로 컬렉션 생성
- Milvus 고유 필터 DSL — SQL-like 표현식 `expr="year > 2020 && source == 'a'"`
- 점수·MMR·영속성

## 언제 쓰나

- 대규모 (수억~수십억) 벡터 프로덕션 — 분산 Milvus
- 로컬·CI 실습에는 Milvus Lite (파일 기반 임베디드)
- GPU 인덱싱(IVF·DiskANN)

## 3.07.1 환경 설정 / 서비스 연결

`pymilvus[milvus_lite]` 만 설치하면 외부 서버 없이 동작한다. probe 셀은 `MilvusClient(uri)` 로 파일을 만들어 헬스체크.

In [ ]:
# !pip install -U langchain-milvus 'pymilvus[milvus_lite]' langchain-openai
# 참고: milvus-lite는 pkg_resources에 의존하므로 setuptools<81 버전이 필요하다.
# !pip install 'setuptools<81'

import os, sys, tempfile
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (임베딩용)"

# probe: Milvus Lite 파일 DB를 만들어 연결 성공을 확인한다.
from pymilvus import MilvusClient
_probe_dir = tempfile.mkdtemp(prefix="milvus_probe_")
_probe_uri = os.path.join(_probe_dir, "probe.db")
_pc = MilvusClient(_probe_uri)
_pc.close()
import shutil; shutil.rmtree(_probe_dir)
print("Milvus Lite OK")

## 3.07.2 기본 사용법

`Milvus.from_documents(docs, embedding, connection_args={'uri': ...}, collection_name=..., drop_old=True)` 로 초기화. `drop_old=True` 는 같은 컬렉션이 있으면 먼저 삭제(데모에 유용).

**주의**: `langchain-milvus 0.3.3` 은 내부적으로 `pymilvus.connections.connect()` 경로를 사용한다. Python 3.14 + 최신 `pymilvus` 조합에서는 로컬 Milvus Lite 경로에서 connection이 등록되지 않아 `ConnectionNotExistException` 이 발생할 수 있다. 3.12 이하 환경에서는 아래 코드가 바로 실행된다.

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Milvus는 대규모 분산 벡터 검색 엔진이다.", metadata={"source": "milvus", "year": 2019}),
    Document(page_content="Milvus Lite는 개발용 파일 기반 모드를 제공한다.", metadata={"source": "milvus", "year": 2024}),
    Document(page_content="DiskANN은 SSD에 최적화된 ANN 인덱스이다.", metadata={"source": "paper", "year": 2019}),
    Document(page_content="IVF_FLAT은 가장 기본적인 Milvus 인덱스 타입이다.", metadata={"source": "docs", "year": 2020}),
]

MILVUS_OK = sys.version_info < (3, 14)

DB_DIR = None
DB_URI = None
store = None

if MILVUS_OK:
    DB_DIR = tempfile.mkdtemp(prefix="milvus_")
    DB_URI = os.path.join(DB_DIR, "demo.db")
    store = Milvus.from_documents(
        documents=docs,
        embedding=embeddings,
        connection_args={"uri": DB_URI},
        collection_name="vs_demo",
        drop_old=True,
    )
    for d in store.similarity_search("분산 벡터 검색", k=3):
        print("-", d.metadata, "|", d.page_content[:40])
else:
    print("Milvus Lite skipped on Python 3.14 (pymilvus connection path incompat) — snippet은 학습용")

## 3.07.3 메타데이터 필터링 — Milvus `expr`

Milvus는 SQL-like **문자열 표현식**을 쓴다.

- 비교: `source == 'milvus'`, `year > 2020`, `year >= 2019`
- 논리: `&&`, `||`
- 포함: `source in ['milvus', 'docs']`
- 문자열: `source like 'mil%'`

LangChain에서는 `similarity_search(query, k, expr='...')` 로 전달한다.

In [ ]:
if store is not None:
    hits = store.similarity_search(
        "인덱스 알고리즘",
        k=5,
        expr="year >= 2019 && source in ['milvus', 'paper']",
    )
    for d in hits:
        print("-", d.metadata, "|", d.page_content[:40])
else:
    print("skipped (Milvus not connected)")

## 3.07.4 점수 포함 · MMR

Milvus `similarity_search_with_score` 는 **L2 거리**(기본 인덱스 COSINE은 유사도)를 반환. 기본은 L2 → 거리가 작을수록 가깝다.

In [ ]:
if store is not None:
    print("--- score ---")
    for doc, score in store.similarity_search_with_score("분산 검색", k=3):
        print(f"{score:.4f}  {doc.metadata['source']}")

    print("\n--- MMR ---")
    for d in store.max_marginal_relevance_search("벡터 인덱스", k=3, fetch_k=4, lambda_mult=0.4):
        print("-", d.metadata["source"], "|", d.page_content[:40])
else:
    print("skipped (Milvus not connected)")

## 3.07.5 영속성 · 재로드

Milvus Lite는 `connection_args['uri']` 경로의 단일 파일(SQLite)을 사용한다. 같은 URI로 `Milvus(...)` 를 다시 만들면 기존 컬렉션을 재사용한다 (`drop_old=False`, 기본값).

In [ ]:
if store is not None:
    reopened = Milvus(
        embedding_function=embeddings,
        collection_name="vs_demo",
        connection_args={"uri": DB_URI},
        drop_old=False,
    )
    for d in reopened.similarity_search("Milvus Lite", k=2):
        print("-", d.metadata, "|", d.page_content[:40])
else:
    print("skipped (Milvus not connected)")

## 3.07.6 원격 Milvus로 이전

프로덕션에서는 `connection_args={'uri': 'http://milvus:19530', 'token': '...'}` 형태로만 바꾸면 된다. 나머지 코드는 그대로. 인덱스 타입·파라미터는 `index_params={'metric_type': 'COSINE', 'index_type': 'HNSW', 'params': {'M': 16, 'efConstruction': 200}}` 로 명시한다.

In [ ]:
# 데모용 파일 정리
if DB_DIR is not None:
    import shutil
    shutil.rmtree(DB_DIR)
    print("cleaned:", DB_DIR)
else:
    print("nothing to clean")

## 체크리스트

- [ ] `pymilvus[milvus_lite]` 설치 + `setuptools<81` (pkg_resources 의존 → Python 3.14 환경에서 필수)
- [ ] Milvus Lite는 **단일 프로세스** 전용 — 멀티 워커는 Milvus Standalone/Distributed
- [ ] 필터 표현식은 SQL-like (`&&`, `||`, `in`, `like`)
- [ ] `drop_old=True` 는 데모용 — 실수로 운영 컬렉션을 지우지 않도록 주의

## 다음

- `08_elasticsearch.ipynb` — BM25와 dense vector를 같은 엔진에서
- `05_retrievers/` — multi-vector retriever

## 참고

- Milvus docs: https://milvus.io/docs
- Milvus Lite: https://milvus.io/docs/milvus_lite.md